# 📖 Lab 4: Rate Limiting at the API Gateway

In the previous labs, we built rate limiting algorithms (Token Bucket, Sliding Window) and made them
distributed with Redis. Now we put it all together — building a **real rate-limited API** that clients
can call over HTTP.

The rate limiter lives at the **API Gateway** level — the front door of our system. It checks every
incoming request **before** the application logic runs. If a client has exceeded their limit, they
get an HTTP `429 Too Many Requests` response immediately. If they're within limits, the request
proceeds normally.

Think of it like a **bouncer at a club** — troublemakers get turned away at the door, not after
they're already inside.

## 🎯 Learning Objectives

- Understand where rate limiters fit in a real API architecture
- See Flask middleware implementing rate limiting
- Send real HTTP requests and observe rate limiting in action
- Read and understand rate limit response headers
- Test different client identification strategies (IP, API key, user ID)

## 🛠️ Setup

This lab uses Docker Compose to run **two services**:

1. **Redis** (port 6381) — Stores rate limit state (token buckets)
2. **Flask API** (port 5050) — Our API server with rate limiting middleware built in

We'll use the `requests` library from this notebook to act as API clients — sending HTTP requests
and observing how the rate limiter responds.

### Start the services

Open a terminal and run:

```bash
cd system-designs/rate-limiter
docker-compose up -d
```

### Select the notebook kernel

In VS Code, click the kernel picker (top-right of this notebook) and select **"Rate Limiter (Python)"**.

If the kernel doesn't appear:
1. Make sure you created the `.venv` and installed dependencies (`uv sync`)
2. Register the kernel: `python -m ipykernel install --user --name=rate-limiter --display-name="Rate Limiter (Python)"`
3. Reload the VS Code window (`Cmd+Shift+P` → "Reload Window")

## 🏗️ Architecture Overview

Here's how all the pieces fit together:

```
┌────────────┐       ┌──────────────────────────────────────┐
│            │       │           Flask API (port 5050)       │
│   Client   │──────>│  ┌──────────────────────────────┐    │
│ (requests  │       │  │   Rate Limit Middleware       │    │
│  library)  │<──429─│  │   1. Identify client (IP/key) │    │
│            │       │  │   2. Check Redis token bucket  │    │
│            │       │  │   3. Allow or reject (429)     │    │
└────────────┘       │  └──────────────┬───────────────┘    │
                     │                 │ (if allowed)        │
                     │  ┌──────────────▼───────────────┐    │
                     │  │   Route Handler              │    │
                     │  │   /api/data, /api/search      │    │
                     │  └──────────────────────────────┘    │
                     └──────────────┬───────────────────────┘
                                    │
                             ┌──────▼──────┐
                             │   Redis     │
                             │ (port 6381) │
                             │ Token state │
                             └─────────────┘
```

**The flow for every request:**

1. Client sends an HTTP request (e.g., `GET /api/data`)
2. Flask middleware intercepts it **before** the route handler
3. Middleware identifies the client (by API key, user ID, or IP address)
4. Middleware runs a Redis Lua script to check/update the token bucket
5. If tokens are available → request proceeds to the route handler → `200 OK`
6. If no tokens left → request is rejected immediately → `429 Too Many Requests`

In [ ]:
import requests
import redis
import time
import json

# Check that the Flask API is running
try:
    resp = requests.get("http://localhost:5050/health")
    print(f"✅ Flask API: {resp.json()}")
except Exception as e:
    print(f"❌ Flask API not running: {e}")
    print("   Run: docker-compose up -d")

# Check that Redis is running
try:
    r = redis.Redis(host="localhost", port=6381, decode_responses=True)
    print(f"✅ Redis: ping={r.ping()}")
except Exception as e:
    print(f"❌ Redis not running: {e}")
    print("   Run: docker-compose up -d")

## 📏 The Rate Limit Rules

Our Flask server has **three tiers** of rate limits, each using the Token Bucket algorithm:

| Rule | Bucket Size (burst) | Refill Rate | Used By |
|------|-------------------|-------------|----------|
| **default** | 10 tokens | 1 token/sec | `/api/data` and most endpoints |
| **search** | 5 tokens | 0.5 tokens/sec | `/api/search` (expensive operation) |
| **premium** | 50 tokens | 10 tokens/sec | `/api/premium` (paid users) |

**What does this mean in practice?**

- **Default**: You can send a burst of 10 requests instantly, then 1 per second after that
- **Search**: Only 5 requests in a burst, and they refill slowly (1 every 2 seconds)
- **Premium**: Generous — 50 burst and 10/sec refill

This is a common real-world pattern: **expensive operations get stricter limits**, while
premium customers get more capacity.

In [ ]:
import requests
import redis
import json

# Clean up previous rate limit state so we start fresh
r = redis.Redis(host="localhost", port=6381, decode_responses=True)
for key in r.keys("ratelimit:*"):
    r.delete(key)
print("🧹 Cleaned up old rate limit keys\n")

# Send a single request to the API
resp = requests.get("http://localhost:5050/api/data")

print(f"Status: {resp.status_code}")
print(f"Body: {json.dumps(resp.json(), indent=2)}")
print()

# Every response includes rate limit headers — let's inspect them
print("Rate Limit Headers:")
for header in ["X-RateLimit-Limit", "X-RateLimit-Remaining", "X-RateLimit-Reset"]:
    value = resp.headers.get(header, "not set")
    print(f"  {header}: {value}")

## 📬 Understanding Rate Limit Headers

Every response from our API includes special headers that tell the client about their rate limit status:

| Header | Meaning | Example |
|--------|---------|----------|
| `X-RateLimit-Limit` | Maximum requests allowed (bucket size) | `10` |
| `X-RateLimit-Remaining` | Requests you have left right now | `7` |
| `X-RateLimit-Reset` | Unix timestamp when your limit refills | `1700000060` |
| `Retry-After` | Seconds to wait before retrying (only on 429) | `2` |

### Why do these matter?

These headers help **well-behaved clients**:

- **Know how many requests they can still make** → avoid wasting requests
- **Know when to slow down** → if `Remaining` is getting low, back off
- **Implement automatic retry logic** → on a 429, wait `Retry-After` seconds, then try again

Think of it like a gas gauge in your car — you don't wait until you run out of gas to find a
station. You check the gauge and plan ahead!

In [ ]:
import requests
import time
import redis

# Clean up so we start with a full bucket
r = redis.Redis(host="localhost", port=6381, decode_responses=True)
for key in r.keys("ratelimit:*"):
    r.delete(key)

print("=== Hitting the Rate Limit ===")
print("Sending 12 requests to /api/data (limit: 10 burst)")
print()

for i in range(12):
    resp = requests.get("http://localhost:5050/api/data")
    remaining = resp.headers.get("X-RateLimit-Remaining", "?")

    if resp.status_code == 200:
        print(f"  Request {i+1:2d}: ✅ {resp.status_code}  (remaining: {remaining})")
    else:
        retry = resp.headers.get("Retry-After", "?")
        print(f"  Request {i+1:2d}: ❌ {resp.status_code}  (remaining: {remaining}, retry-after: {retry}s)")

print()
print("💡 After hitting the limit, wait for tokens to refill.")
print("   The server adds 1 token/second.")

## ⏳ Waiting for Token Refill

We just used up all 10 tokens in our bucket. But the Token Bucket algorithm **refills over time** —
our default rule adds 1 token per second.

So if we wait 3 seconds, we should have ~3 tokens available again. Let's test that!

In [ ]:
import time
import requests

print("Waiting 3 seconds for tokens to refill...")
time.sleep(3)

print("\nSending 3 more requests:")
for i in range(3):
    resp = requests.get("http://localhost:5050/api/data")
    remaining = resp.headers.get("X-RateLimit-Remaining", "?")
    status = "✅" if resp.status_code == 200 else "❌"
    print(f"  Request {i+1}: {status} {resp.status_code}  (remaining: {remaining})")

## 🆔 Client Identification

How does the server know **which client** is making a request? It needs to identify each client
so it can track their individual rate limit bucket.

Our Flask server checks for identity in this priority order:

1. **`X-API-Key` header** → identified as `apikey:your-key`
2. **`X-User-Id` header** → identified as `user:your-id`
3. **IP address** (fallback) → identified as `ip:172.17.0.1` (your Docker IP)

**The key insight**: Each identity gets its own **separate** rate limit bucket!

This means:
- Alice (with API key `alice-key`) has her own 10 tokens
- Bob (with user ID `bob`) has his own 10 tokens
- Even if Alice is rate-limited, Bob can still make requests

In [ ]:
import requests
import redis

# Clean up so everyone starts fresh
r = redis.Redis(host="localhost", port=6381, decode_responses=True)
for key in r.keys("ratelimit:*"):
    r.delete(key)

print("=== Different Clients, Separate Limits ===\n")

# Alice uses an API key
print("Alice (API key: alice-key):")
for i in range(3):
    resp = requests.get("http://localhost:5050/api/data", headers={"X-API-Key": "alice-key"})
    print(f"  Request {i+1}: {resp.status_code} (remaining: {resp.headers.get('X-RateLimit-Remaining')})")

print()

# Bob uses a user ID
print("Bob (User ID: bob):")
for i in range(3):
    resp = requests.get("http://localhost:5050/api/data", headers={"X-User-Id": "bob"})
    print(f"  Request {i+1}: {resp.status_code} (remaining: {resp.headers.get('X-RateLimit-Remaining')})")

print()

# Let's peek inside Redis to see the separate buckets
print("Redis keys (each client has its own bucket):")
for key in sorted(r.keys("ratelimit:*")):
    key_type = r.type(key)
    if key_type == "hash":
        data = r.hgetall(key)
    else:
        data = r.get(key)
    print(f"  🔑 {key}: {data}")

## 🔀 Endpoint-Specific Limits

Not all endpoints are created equal. Some are cheap (reading cached data), while others are
expensive (running a search query across millions of records).

Our API has different rate limits for different endpoints:

- **`/api/data`** — Default limit: **10 burst**, 1/sec refill
- **`/api/search`** — Stricter limit: **5 burst**, 0.5/sec refill

This is very common in production APIs. For example, Twitter's API has different limits for
reading tweets vs. searching tweets vs. posting tweets.

Let's send requests to both endpoints and see how the limits differ!

In [ ]:
import requests
import redis

# Clean up
r = redis.Redis(host="localhost", port=6381, decode_responses=True)
for key in r.keys("ratelimit:*"):
    r.delete(key)

print("=== Endpoint-Specific Rate Limits ===")
print()

# Test /api/data — default limit (10 burst, 1/sec refill)
print("/api/data — Default limit (10 burst, 1/sec refill):")
for i in range(6):
    resp = requests.get("http://localhost:5050/api/data", headers={"X-API-Key": "test"})
    status = "✅" if resp.status_code == 200 else "❌"
    print(f"  Request {i+1}: {status} (remaining: {resp.headers.get('X-RateLimit-Remaining')})")

print()

# Test /api/search — stricter limit (5 burst, 0.5/sec refill)
print("/api/search — Stricter limit (5 burst, 0.5/sec refill):")
for i in range(6):
    resp = requests.get("http://localhost:5050/api/search?q=hello", headers={"X-API-Key": "test"})
    status = "✅" if resp.status_code == 200 else "❌"
    remaining = resp.headers.get("X-RateLimit-Remaining", "?")
    print(f"  Request {i+1}: {status} (remaining: {remaining})")

print()
print("💡 The search endpoint hit its limit after 5 requests, while /api/data still had room!")

## 🔍 Anatomy of a 429 Response

When a client exceeds their rate limit, the server sends back an **HTTP 429 Too Many Requests**
response. Let's look closely at exactly what that looks like — the status code, headers, and body.

In [ ]:
import requests
import json
import redis

# Clean up
r = redis.Redis(host="localhost", port=6381, decode_responses=True)
for key in r.keys("ratelimit:*"):
    r.delete(key)

# Exhaust the rate limit by sending 10 requests
print("Exhausting rate limit...")
for i in range(10):
    requests.get("http://localhost:5050/api/data", headers={"X-API-Key": "inspect-test"})

# Now the next request should be rejected with a 429
resp = requests.get("http://localhost:5050/api/data", headers={"X-API-Key": "inspect-test"})

print(f"\n=== HTTP 429 Response Details ===\n")
print(f"Status Code: {resp.status_code}")
print(f"Status Text: Too Many Requests")
print()

# Show rate limit related headers
print("Headers:")
for header, value in resp.headers.items():
    if "ratelimit" in header.lower() or "retry" in header.lower():
        print(f"  📋 {header}: {value}")

print()
print("Body:")
print(f"  {json.dumps(resp.json(), indent=2)}")

print()
print("💡 A well-behaved client would:")
print(f"   1. See the 429 status code")
print(f"   2. Read Retry-After: {resp.headers.get('Retry-After', '?')} seconds")
print(f"   3. Wait that long before retrying")
print(f"   4. Use X-RateLimit-Remaining to slow down before hitting the limit")

## 🌍 Real-World Rate Limiting Patterns

Our lab demonstrates the fundamentals, but production systems go much further. Here are
patterns you'll see in real-world APIs:

### 1. Layered Limits
Multiple rate limits applied simultaneously:
- **Per-user**: 100 req/min per user
- **Per-IP**: 1000 req/min per IP (catches abuse from shared IPs)
- **Per-endpoint**: 20 req/min for search, 100 req/min for reads
- **Global**: 10,000 req/min across all users (protects the system)

The **most restrictive** rule wins.

### 2. Tiered Limits
Different limits based on subscription level:
- Free tier: 100 req/min
- Pro tier: 1,000 req/min
- Enterprise: 10,000 req/min (or custom)

### 3. Fail Closed vs. Fail Open
What happens when Redis goes down?
- **Fail closed** (our approach): Reject all requests → safe but causes outage
- **Fail open**: Allow all requests → keeps working but no rate limiting

Most production systems fail open with monitoring alerts.

### 4. Geographic Distribution
Place Redis near your users for low latency. If your API serves users in the US, Europe, and Asia,
you might have Redis clusters in each region.

### 5. Redis Sharding
For 1M+ requests/sec, a single Redis instance isn't enough. Redis Cluster can shard rate limit
keys across multiple nodes.

### Real-World Examples

| Service | Rate Limit |
|---------|------------|
| GitHub API | 5,000 req/hour (authenticated) |
| Twitter API | 300 req/15 min (per endpoint) |
| Stripe API | 100 req/sec (per API key) |
| OpenAI API | Varies by model and tier |
| AWS API Gateway | 10,000 req/sec (configurable) |

In [ ]:
import redis

# Clean up all rate limit keys from Redis
r = redis.Redis(host="localhost", port=6381, decode_responses=True)
for key in r.keys("ratelimit:*"):
    r.delete(key)
print("✅ Cleaned up all rate limit keys")
print()
print("To stop the services:")
print("  cd system-designs/rate-limiter")
print("  docker-compose down")

## 📝 Key Takeaways

1. **Rate limiters belong at the API Gateway** — the front door of your system
2. **Flask middleware** checks every request before the route handler runs
3. **HTTP 429 + rate limit headers** tell clients exactly when to retry
4. **Different endpoints can have different limits** (search vs. regular)
5. **Each client identity** (IP, API key, user ID) gets its own bucket
6. **Redis** makes it work across multiple servers
7. **Real APIs layer multiple rules** and enforce the most restrictive

## 🎉 Lab Complete!

Congratulations! You've now built rate limiting **from the ground up** across four labs:

1. **Lab 1**: Token Bucket algorithm (pure Python)
2. **Lab 2**: Sliding Window Counter (fixed vs. sliding windows)
3. **Lab 3**: Distributed rate limiting with Redis (Lua scripts for atomicity)
4. **Lab 4**: Rate limiting at the API Gateway (Flask middleware, HTTP 429)

### 🧠 Key System Design Takeaways

- **Place the rate limiter at the API Gateway level** — it's the first thing that runs
- **Use Token Bucket** for simple, burst-friendly rate limiting
- **Use Redis** for distributed state with Lua scripts for atomicity
- **Always include rate limit headers** in responses (clients need them!)
- **Scale with Redis Cluster** sharding and master-replica failover

You now have a solid understanding of how rate limiters work in production systems. When you
see rate limiting mentioned in system design interviews, you'll know exactly how it's built! 🚀